## 1. Téléchargement de vos Datasets Kaggle dans `/workspace/data_raw/<source>/`

Renseignez dans la liste `KAGGLE_DATASETS` un ou plusieurs identifiants Kaggle (ex. `["auteur1/dataset-principal", "auteur2/meteo-ou-referentiel"]`), ou déposez directement vos fichiers/sous-dossiers dans `data_raw/`.
- Chaque dataset est isolé dans son propre sous-dossier `data_raw/<nom-dataset>/` pour éviter toute collision de noms (`train.csv`, `metadata.csv`, etc.) entre plusieurs sources.
- **Critère important :** Vérifiez qu'au moins un fichier dépasse **20 à 32 Mo** afin d'observer sa division en plusieurs blocs HDFS de **16 Mo** !

In [ ]:
import re
import shutil
from pathlib import Path

DATA_RAW = Path("/workspace/data_raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)
SUPPORTED_EXTS = {".csv", ".parquet", ".json", ".jsonl", ".tsv"}

KAGGLE_DATASETS: list[str] = [
    "gauravmehta13/airline-flight-delays",
]
AIRLINE_FILES = ("flights.csv", "airlines.csv", "airports.csv", "cancellation_codes.csv")

for slug in KAGGLE_DATASETS:
    slug = slug.strip()
    if not slug:
        continue
    if slug == "gauravmehta13/airline-flight-delays" and all((DATA_RAW / name).is_file() for name in AIRLINE_FILES):
        print(f"Dataset {slug} déjà présent dans {DATA_RAW}.")
        continue
    subdir = re.sub(r"[^a-zA-Z0-9._-]+", "_", slug.split("/")[-1]).strip("_") or "dataset"
    target_dir = DATA_RAW / subdir
    existing = [f for f in target_dir.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS] if target_dir.exists() else []
    if not existing:
        import kagglehub
        print(f"Téléchargement de {slug} -> {target_dir}/ ...")
        cache_dir = Path(kagglehub.dataset_download(slug))
        target_dir.mkdir(parents=True, exist_ok=True)
        for src_file in sorted(cache_dir.rglob("*")):
            if src_file.is_file() and src_file.suffix.lower() in SUPPORTED_EXTS:
                dest_file = target_dir / src_file.relative_to(cache_dir)
                dest_file.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src_file, dest_file)

local_files = sorted([f for f in DATA_RAW.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

assert len(local_files) > 0, (
    "Aucun fichier détecté dans /workspace/data_raw/. "
    "Renseignez `KAGGLE_DATASETS` ci-dessus ou utilisez `task data:download DATASET='auteur1/ds1,auteur2/ds2'`."
)

print(f"Inventaire local multi-sources dans {DATA_RAW} ({len(local_files)} fichier(s)) :")
for f in local_files:
    rel = f.relative_to(DATA_RAW)
    size_mb = f.stat().st_size / (1024 * 1024)
    expected_blocks = max(1, int(-(-f.stat().st_size // (16 * 1024 * 1024))))
    print(f" - {str(rel):<55} {size_mb:>7.2f} Mo  (~{expected_blocks} bloc(s) HDFS de 16 Mo)")

Dataset gauravmehta13/airline-flight-delays déjà présent dans /workspace/data_raw.
Inventaire local multi-sources dans /workspace/data_raw (4 fichier(s)) :
 - airlines.csv                                               0.00 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - airports.csv                                               0.02 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - cancellation_codes.csv                                     0.00 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - flights.csv                                              564.96 Mo  (~36 bloc(s) HDFS de 16 Mo)


## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark pointe automatiquement vers `hdfs://namenode:9000` avec **Delta Lake** activé.

### Exercice 2.1 — Initialiser `SparkSession` et contrôler la configuration HDFS
1. Créez une session Spark nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Récupérez via `spark.conf.get(...)` la valeur de `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"`.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# TODO : Initialiser la SparkSession
spark = (
    SparkSession.builder
    .appName("HDFS Ingestion Bronze")
    .getOrCreate()
)

assert spark is not None, "TODO : Initialisez `spark` avec SparkSession.builder..."

# TODO : Lire les 3 paramètres HDFS depuis `spark.conf.get(...)`
default_fs = spark.conf.get("spark.hadoop.fs.defaultFS")
block_size = spark.conf.get("spark.hadoop.dfs.blocksize")
replication = spark.conf.get("spark.hadoop.dfs.replication")

print("Spark version       :", spark.version)
print("Default FileSystem  :", default_fs)
print("HDFS Block Size (o) :", block_size)
print("HDFS Replication    :", replication)

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ce5ff6d7-f2c8-4b1d-8a14-468e4a6d4722;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

Spark version       : 4.2.0
Default FileSystem  : hdfs://namenode:9000
HDFS Block Size (o) : 16777216
HDFS Replication    : 2


## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante téléverse vos fichiers bruts de `/workspace/data_raw` vers `/lakehouse/bronze/raw/` sur HDFS, puis interroge le NameNode pour afficher la localisation exacte de chaque bloc de 16 Mo sur les 3 DataNodes (`datanode1`, `datanode2`, `datanode3`).

In [3]:
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw")
fs.mkdirs(raw_hdfs_dir)

for file_path in local_files:
    rel_posix = file_path.relative_to(DATA_RAW).as_posix()
    src = PathHDFS(f"file://{file_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw/{rel_posix}")
    fs.mkdirs(dst.getParent())
    fs.copyFromLocalFile(False, True, src, dst)
    print(f"Téléversé vers HDFS : {dst}")

Téléversé vers HDFS : /lakehouse/bronze/raw/airlines.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/airports.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/cancellation_codes.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/flights.csv


In [4]:
# Inspection récursive du découpage en blocs de 16 Mo sur l'ensemble des sources ingérées
remote_iter = fs.listFiles(raw_hdfs_dir, True)
statuses = []
while remote_iter.hasNext():
    statuses.append(remote_iter.next())
statuses.sort(key=lambda s: s.getLen(), reverse=True)
largest_hdfs_path = statuses[0].getPath().toString()

for status in statuses:
    path = status.getPath()
    rel_hdfs = path.toString().split("/lakehouse/bronze/raw/", 1)[-1]
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {rel_hdfs} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

Fichier : flights.csv (564.96 Mo | repl=2)
   Bloc #0 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #1 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #2 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #3 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #4 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #5 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #6 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #7 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
   Bloc #8 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #9 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #10 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #11 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
   Bloc #12 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #13 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #14 (16.00 Mo) -> DataNodes : ['datanode2', 'data

### Questions d'analyse (à vérifier aussi sur http://localhost:9870)
1. Comparez sur l'interface Web du NameNode (`Utilities > Browse the file system`) le nombre de blocs de votre fichier le plus volumineux avec sa taille divisée par 16 Mo.

Le fichier le plus volumineux est flights.csv, avec une taille de 564,96 Mio. La taille maximale d'un bloc HDFS est configurée à 16 Mio (16 777 216 octets).
En divisant 564,96 par 16, nous obtenons 35,31, soit 36 blocs après arrondi à l'entier supérieur.
Les résultats confirment ce calcul : HDFS a créé 36 blocs, dont 35 blocs de 16 Mio et un dernier bloc de 4,96 Mio.
Le découpage observé est donc cohérent avec la configuration HDFS.

2. Un fichier de 3 Mo bloque-t-il réellement 16 Mo d'espace disque sur les DataNodes ? Pourquoi ?

Non. La taille de 16 Mo représente la taille maximale configurée pour un bloc HDFS. Un fichier de 3 Mo occupe environ 3 Mo par copie, et non 16 Mo. Avec un facteur de réplication de 2, il occupe donc environ 6 Mo au total.

3. Pourquoi avoir configuré `dfs.replication = 2` sur un cluster à `3` DataNodes ?

Le facteur de réplication de 2 permet de conserver deux copies de chaque bloc sur des DataNodes différents ce qui permet d'avoir une copie disponible si un DataNode tombe en panne. Le NameNode peut ensuite demander la création d'une nouvelle copie sur un DataNode encore actif pour rétablir le facteur de réplication. 


---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans `config/hdfs-site.xml`, le délai de détection d'un DataNode mort a été abaissé de 10m30s à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole à réaliser depuis votre terminal :
1. Inspectez l'état initial des blocs : `task hdfs:fsck`
2. Arrêtez brutalement le DataNode 2 :
   ```bash
   task hdfs:crash-dn2   # équivalent à : docker stop datanode2
   ```
3. Observez l'onglet **Datanodes** sur `http://localhost:9870` pendant ~50s : dès que `datanode2` passe à **Dead**, le NameNode détecte les blocs sous-répliqués et ordonne leur copie automatique vers le 3e DataNode survivant !

### Exercice 4.1 — Vérifier la continuité de service et la *Data Locality*
Pendant que `datanode2` est éteint, lisez votre plus gros fichier (`largest_hdfs_path`) avec PySpark, affichez `df_check.count()` ainsi que `df_check.rdd.getNumPartitions()`, et expliquez le lien avec le nombre de blocs HDFS.

Le fichier flights.csv, d'une taille de 564,96 Mio, est découpé en 36 blocs HDFS de 16 Mio maximum, avec un facteur de réplication de 2.
Lors d'une lecture précédente sans option d'en-tête, Spark a compté 5 819 080 lignes, dont l'en-tête, et créé 16 partitions. En excluant l'en-tête, le fichier contient 5 819 079 vols ; le nombre de partitions peut varier selon la configuration.
Le nombre de partitions Spark est donc différent du nombre de blocs HDFS. Cette différence s'explique par leurs rôles distincts : HDFS découpe les fichiers en blocs pour les stocker et les répliquer sur les DataNodes, tandis que Spark crée des partitions pour organiser le traitement des données en parallèle. Une partition Spark peut regrouper les données de plusieurs blocs HDFS. Il n'existe donc pas de correspondance systématique entre les blocs HDFS et les partitions Spark.
Lors du crash-test du 17 novembre 2026, le rapport HDFS indiquait 2 DataNodes vivants, 1 mort, 40 blocs sous-répliqués et aucun bloc manquant. Spark a lu 5 819 079 vols pendant l'arrêt de datanode2.
Lors d'un second test, 20 blocs Bronze possédaient une réplique sur datanode2 avant son arrêt. Le rapport a ensuite indiqué 2 DataNodes vivants, datanode2 mort, aucun bloc manquant et aucun bloc sous-répliqué avant son redémarrage : la réplication a donc été rétablie sur les nœuds restants.
Après redémarrage, le cluster indiquait 3 DataNodes vivants, aucun mort et aucun bloc sous-répliqué.


*(Pensez ensuite à rallumer le nœud avec `task hdfs:restore-dn2` !)*

In [5]:
print("Fichier cible du test de résilience :", largest_hdfs_path)

df_check = spark.read.option("header", "true").csv(largest_hdfs_path)

print(f"Nombre total de lignes lues       : {df_check.count():,}")
print(f"Nombre de partitions Spark créées : {df_check.rdd.getNumPartitions()}")

Fichier cible du test de résilience : hdfs://namenode:9000/lakehouse/bronze/raw/flights.csv


Nombre total de lignes lues       : 5,819,079
Nombre de partitions Spark créées : 16


---
## 5. Construction de votre Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir vos fichiers bruts (mono ou multi-datasets) en tables Delta Lake Bronze
Dans une architecture **Médaillon multi-sources**, la couche **Bronze** conserve chaque table brute fidèle à sa source tout en l'enrichissant de métadonnées de lignage (*lineage*) indispensables avant les jointures du Jour 2 :
- `_ingested_at` : `F.current_timestamp()`
- `_source_dataset` : `F.lit(dataset_name)` *(identifie le dataset d'origine)*
- `_source_file` : `F.col("_metadata.file_path")`

> **Attention aux noms de colonnes Delta/Parquet :** Si vos fichiers CSV bruts contiennent des espaces ou caractères spéciaux dans les en-têtes, normalisez-les simplement en `snake_case` (ex. `.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])`).

La cellule ci-dessous ingère les fichiers CSV et Parquet dans `/lakehouse/bronze/<nom_table>`. Elle refuse les autres formats et les noms de table en double pour éviter toute ingestion incorrecte ou écrasement accidentel.

**Contrôle Spark Web UI :** le 17 novembre 2026, la session `J1_HDFS_Ingestion_Bronze` affichait 47 jobs réussis et aucun job en échec. Les stages comprenaient 47 étapes terminées et 33 étapes ignorées.

In [6]:
unsupported_files = [f for f in local_files if f.suffix.lower() not in {".csv", ".parquet"}]
if unsupported_files:
    raise ValueError(f"Formats Bronze non pris en charge : {unsupported_files}")
table_names = [f.stem.lower().replace(" ", "_") for f in local_files]
if len(set(table_names)) != len(table_names):
    raise ValueError("Plusieurs fichiers produisent le même nom de table Bronze.")

for f in local_files:
    rel = f.relative_to(DATA_RAW)
    dataset_name = rel.parts[0] if len(rel.parts) > 1 else ("airline_flight_delays" if f.name in AIRLINE_FILES else "default")
    table_name = f.stem.lower().replace(" ", "_")
    raw_path = f"/lakehouse/bronze/raw/{rel.as_posix()}"
    bronze_delta_path = f"/lakehouse/bronze/{table_name}"

    df_raw = (
        spark.read.parquet(raw_path)
        if f.suffix.lower() == ".parquet"
        else spark.read.option("header", "true").option("inferSchema", "true").option("multiLine", "true").option("escape", '"').csv(raw_path)
    )
    df_raw = (
        df_raw.toDF(*[c.strip().lower().replace(" ", "_") for c in df_raw.columns])
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_dataset", F.lit(dataset_name))
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    df_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze_delta_path)
    cnt = spark.read.format("delta").load(bronze_delta_path).count()
    print(f"[BRONZE OK] {bronze_delta_path:<55} (source={dataset_name}) -> {cnt:>10,} lignes")

26/11/19 11:03:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[BRONZE OK] /lakehouse/bronze/airlines                              (source=airline_flight_delays) ->         14 lignes


[BRONZE OK] /lakehouse/bronze/airports                              (source=airline_flight_delays) ->        322 lignes


[BRONZE OK] /lakehouse/bronze/cancellation_codes                    (source=airline_flight_delays) ->          4 lignes


[BRONZE OK] /lakehouse/bronze/flights                               (source=airline_flight_delays) ->  5,819,079 lignes


## 5.2 Ingestion des observations météo NOAA (2015)

Cette étape télécharge les observations horaires **NOAA ISD-Lite** des stations situées à moins de 25 km des aéroports. Les observations sont en UTC; la table de correspondance conserve le fuseau IANA de chaque aéroport pour convertir correctement les heures locales des vols. Les lignes brutes sont conservées avec les mesures converties en unités physiques dans Bronze. Les aéroports sans station proche ou sans coordonnées sont signalés et restent sans météo.

Exécutez ces cellules après la construction des tables Bronze Kaggle et avant J2. Source publique : [NOAA ISD-Lite](https://noaa-isd-pds.s3.amazonaws.com/isd-lite/isd-lite-format.txt).


In [7]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("timezonefinder") is None:
    subprocess.run(
        ["uv", "pip", "install", "--python", sys.executable, "timezonefinder>=9.2,<10"],
        check=True,
    )


In [8]:
import csv
import gzip
import io
import math
import re
import tempfile
import time
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from pathlib import Path

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, StructField, StructType
from pyspark.storagelevel import StorageLevel
from timezonefinder import TimezoneFinder

NOAA_YEAR = 2015
NOAA_BUCKET = "https://noaa-isd-pds.s3.amazonaws.com"
NOAA_HISTORY_URL = f"{NOAA_BUCKET}/isd-history.csv"
NOAA_MAX_DISTANCE_KM = 25.0
NOAA_BRONZE_WEATHER_PATH = f"/lakehouse/bronze/noaa_weather_{NOAA_YEAR}"
NOAA_BRONZE_STATIONS_PATH = "/lakehouse/bronze/weather_station_map"


def fetch_noaa_bytes(url, attempts=3):
    request = Request(url, headers={"User-Agent": "airline-datas-notebook/1.0"})
    for attempt in range(attempts):
        try:
            with urlopen(request, timeout=60) as response:
                return response.read()
        except HTTPError as error:
            if error.code == 404 or attempt == attempts - 1:
                raise
        except (TimeoutError, URLError):
            if attempt == attempts - 1:
                raise
        time.sleep(2**attempt)
    raise RuntimeError(f"Impossible de télécharger {url}")


def available_isd_lite_stations(year):
    namespace = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}
    prefix = f"isd-lite/data/{year}/"
    continuation_token = None
    station_ids = set()
    while True:
        params = {"list-type": "2", "prefix": prefix, "max-keys": "1000"}
        if continuation_token:
            params["continuation-token"] = continuation_token
        root = ET.fromstring(fetch_noaa_bytes(f"{NOAA_BUCKET}/?{urlencode(params)}"))
        for item in root.findall("s3:Contents", namespace):
            key = item.findtext("s3:Key", default="", namespaces=namespace)
            filename = key.rsplit("/", 1)[-1]
            match = re.fullmatch(r"(\d{6}-\d{5})-\d{4}\.gz", filename)
            if match:
                station_ids.add(match.group(1))
        if root.findtext("s3:IsTruncated", default="false", namespaces=namespace) != "true":
            return station_ids
        continuation_token = root.findtext("s3:NextContinuationToken", namespaces=namespace)
        if not continuation_token:
            raise RuntimeError("Le catalogue NOAA est tronqué sans jeton de continuation.")


def distance_km(lat1, lon1, lat2, lon2):
    lat1_rad, lat2_rad = math.radians(lat1), math.radians(lat2)
    delta_lat = lat2_rad - lat1_rad
    delta_lon = math.radians(lon2 - lon1)
    value = math.sin(delta_lat / 2) ** 2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(delta_lon / 2) ** 2
    return 2 * 6371.0088 * math.asin(math.sqrt(value))


def scale_noaa_value(index, divisor=1.0):
    raw_value = F.col("parts")[index].cast("int")
    return F.when(raw_value == -9999, F.lit(None).cast("double")).otherwise(raw_value.cast("double") / divisor)


# Le catalogue historique permet de choisir uniquement des stations ayant un fichier ISD-Lite pour 2015.
available_station_ids = available_isd_lite_stations(NOAA_YEAR)
with urlopen(Request(NOAA_HISTORY_URL, headers={"User-Agent": "airline-datas-notebook/1.0"}), timeout=60) as response:
    station_rows = list(csv.DictReader(io.TextIOWrapper(response, encoding="utf-8")))

first_day, last_day = f"{NOAA_YEAR}0101", f"{NOAA_YEAR}1231"
station_catalog = []
for row in station_rows:
    station_id = f"{row['USAF']}-{row['WBAN']}"
    if station_id not in available_station_ids or not row["LAT"] or not row["LON"]:
        continue
    if row["BEGIN"] > last_day or row["END"] < first_day:
        continue
    try:
        station_latitude, station_longitude = float(row["LAT"]), float(row["LON"])
    except ValueError:
        continue
    station_catalog.append({
        "station_id": station_id,
        "station_name": row["STATION NAME"],
        "latitude": station_latitude,
        "longitude": station_longitude,
    })
if not station_catalog:
    raise RuntimeError(f"Aucune station NOAA ISD-Lite valide pour {NOAA_YEAR}.")

# Le référentiel Kaggle fournit les coordonnées; TimezoneFinder retourne un identifiant IANA utilisable par Spark.
airport_rows = (
    spark.read.format("delta").load("/lakehouse/bronze/airports")
    .select("iata_code", "latitude", "longitude").dropDuplicates(["iata_code"]).collect()
)
timezone_finder = TimezoneFinder(in_memory=True)
airport_station_rows = []
for airport in airport_rows:
    airport_code = str(airport.iata_code).strip().upper()
    try:
        airport_latitude = float(airport.latitude)
        airport_longitude = float(airport.longitude)
    except (TypeError, ValueError):
        airport_latitude = airport_longitude = None
    airport_timezone = (
        timezone_finder.timezone_at(lng=airport_longitude, lat=airport_latitude)
        if airport_latitude is not None and airport_longitude is not None
        else None
    )
    nearest = None
    nearest_distance = None
    if airport_latitude is not None and airport_longitude is not None:
        for station in station_catalog:
            distance = distance_km(
                airport_latitude, airport_longitude, station["latitude"], station["longitude"]
            )
            if nearest_distance is None or distance < nearest_distance:
                nearest, nearest_distance = station, distance
    if nearest_distance is not None and nearest_distance > NOAA_MAX_DISTANCE_KM:
        nearest = nearest_distance = None
    airport_station_rows.append((
        airport_code,
        nearest["station_id"] if nearest else None,
        nearest["station_name"] if nearest else None,
        float(nearest_distance) if nearest_distance is not None else None,
        airport_timezone,
    ))

station_map_schema = StructType([
    StructField("airport_code", StringType(), False),
    StructField("station_id", StringType(), True),
    StructField("station_name", StringType(), True),
    StructField("station_distance_km", DoubleType(), True),
    StructField("airport_timezone", StringType(), True),
])
df_weather_station_map = spark.createDataFrame(airport_station_rows, station_map_schema)
if df_weather_station_map.groupBy("airport_code").count().filter(F.col("count") > 1).limit(1).count():
    raise ValueError("Le mapping aéroport/NOAA contient des codes aéroport en double.")

station_ids = sorted({row[1] for row in airport_station_rows if row[1]})
if not station_ids:
    raise RuntimeError("Aucun aéroport ne possède de station NOAA dans le rayon de 25 km.")

with tempfile.TemporaryDirectory(prefix="noaa-isd-lite-") as temp_name:
    temp_dir = Path(temp_name)

    def download_station(station_id):
        url = f"{NOAA_BUCKET}/isd-lite/data/{NOAA_YEAR}/{station_id}-{NOAA_YEAR}.gz"
        target = temp_dir / f"{station_id}-{NOAA_YEAR}.gz"
        target.write_bytes(fetch_noaa_bytes(url))
        return target

    with ThreadPoolExecutor(max_workers=8) as executor:
        station_files = list(executor.map(download_station, station_ids))

    raw_weather = spark.read.text([path.resolve().as_uri() for path in station_files]).withColumn(
        "source_file", F.input_file_name()
    )
    raw_weather = raw_weather.withColumn("parts", F.split(F.trim(F.col("value")), r"\s+"))
    malformed_count = raw_weather.filter(F.size("parts") != 12).count()
    if malformed_count:
        raise ValueError(f"{malformed_count} lignes NOAA ne contiennent pas les 12 champs ISD-Lite attendus.")

    spark.conf.set("spark.sql.session.timeZone", "UTC")
    precipitation_1h_raw = F.col("parts")[10].cast("int")
    precipitation_6h_raw = F.col("parts")[11].cast("int")
    df_noaa_weather = raw_weather.select(
        F.regexp_extract("source_file", r"([0-9]{6}-[0-9]{5})-2015\.gz$", 1).alias("station_id"),
        F.to_timestamp(
            F.concat_ws(" ", F.col("parts")[0], F.col("parts")[1], F.col("parts")[2], F.col("parts")[3]),
            "yyyy MM dd HH",
        ).alias("observation_utc"),
        F.col("value").alias("raw_isd_lite_line"),
        scale_noaa_value(4, 10.0).alias("air_temperature_c"),
        scale_noaa_value(5, 10.0).alias("dew_point_c"),
        scale_noaa_value(6, 10.0).alias("sea_level_pressure_hpa"),
        scale_noaa_value(7).alias("wind_direction_deg"),
        scale_noaa_value(8, 10.0).alias("wind_speed_mps"),
        scale_noaa_value(9).alias("sky_cover_code"),
        F.when(precipitation_1h_raw == -9999, F.lit(None).cast("double"))
        .when(precipitation_1h_raw == -1, F.lit(0.0))
        .otherwise(precipitation_1h_raw.cast("double") / 10.0).alias("precipitation_1h_mm"),
        F.when(precipitation_6h_raw == -9999, F.lit(None).cast("double"))
        .when(precipitation_6h_raw == -1, F.lit(0.0))
        .otherwise(precipitation_6h_raw.cast("double") / 10.0).alias("precipitation_6h_mm"),
        (precipitation_1h_raw == -1).alias("precipitation_1h_trace"),
        (precipitation_6h_raw == -1).alias("precipitation_6h_trace"),
    ).withColumn("_ingested_at", F.current_timestamp()) \
     .withColumn("_source_dataset", F.lit("NOAA_ISD_Lite_2015")) \
     .withColumn("_source_file", F.col("station_id")) \
     .persist(StorageLevel.DISK_ONLY)

    try:
        invalid_weather_keys = df_noaa_weather.filter(
            F.col("station_id").isNull() | F.col("observation_utc").isNull()
        ).limit(1).count()
        if invalid_weather_keys:
            raise ValueError("Des observations NOAA ont une clé station/heure invalide.")
        duplicate_weather_hours = (
            df_noaa_weather.groupBy("station_id", "observation_utc")
            .count().filter(F.col("count") > 1).limit(1).count()
        )
        if duplicate_weather_hours:
            raise ValueError("Les observations NOAA contiennent plusieurs lignes par station et heure UTC.")

        df_noaa_weather.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
            NOAA_BRONZE_WEATHER_PATH
        )
        df_weather_station_map.withColumn("_ingested_at", F.current_timestamp()) \
            .withColumn("_source_dataset", F.lit("NOAA_ISD_Lite_2015")) \
            .withColumn("_source_file", F.lit("isd-history.csv + airports.csv")) \
            .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
                NOAA_BRONZE_STATIONS_PATH
            )
        print(f"[BRONZE OK] {NOAA_BRONZE_WEATHER_PATH} -> {df_noaa_weather.count():,} observations horaires")
        print(f"[BRONZE OK] {NOAA_BRONZE_STATIONS_PATH} -> {df_weather_station_map.count():,} aéroports")
        print(f"Stations NOAA uniques : {len(station_ids)}")
        df_weather_station_map.agg(
            F.sum(F.col("station_id").isNotNull().cast("int")).alias("airports_with_weather_station"),
            F.sum(F.col("airport_timezone").isNotNull().cast("int")).alias("airports_with_timezone"),
        ).show(truncate=False)
    finally:
        df_noaa_weather.unpersist()


26/11/19 11:05:12 WARN MemoryManager: Total allocation exceeds 95.00% (2,040,109,440 bytes) of heap memory
Scaling row group sizes to 95.00% for 16 writers
26/11/19 11:05:20 WARN MemoryManager: Total allocation exceeds 95.00% (2,040,109,440 bytes) of heap memory
Scaling row group sizes to 95.00% for 16 writers


[BRONZE OK] /lakehouse/bronze/noaa_weather_2015 -> 2,707,427 observations horaires
[BRONZE OK] /lakehouse/bronze/weather_station_map -> 322 aéroports
Stations NOAA uniques : 317
+-----------------------------+----------------------+
|airports_with_weather_station|airports_with_timezone|
+-----------------------------+----------------------+
|317                          |319                   |
+-----------------------------+----------------------+



In [9]:
# Auto-contrôle de fin de Jour 1 : au moins une table Delta valide dans /lakehouse/bronze/
bronze_root = PathHDFS("/lakehouse/bronze")
bronze_delta_tables = [
    s.getPath().getName()
    for s in fs.listStatus(bronze_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
missing_tables = set(table_names) - set(bronze_delta_tables)
assert not missing_tables, f"Tables Delta manquantes : {sorted(missing_tables)}"
print("Tables Delta Bronze validées :", bronze_delta_tables)

Tables Delta Bronze validées : ['airlines', 'airports', 'cancellation_codes', 'flights', 'noaa_weather_2015', 'weather_station_map']
